In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
import sys
import os
import plotfancy as pf
from matplotlib.patches import Circle
from astropy.visualization import ZScaleInterval
from astropy.table import Table
from types import SimpleNamespace
import re
from astropy.io import ascii
from astropy import units as u
from astropy.table import Table, vstack, hstack
from astropy.modeling import models, fitting

import os
from io import StringIO
from astropy.table import vstack

# from calculate_jiang19_metallicity import calculate_metallicity_jiang19 as cjm19
sys.path.append('../../../')
import src.ifu_tools.line_ratios as lr
# import src.ifu_tools.ifutools as ift
from src.ifu_tools.ifutools import museCube
# import line_ratios as lr

import logging 
logging.getLogger('mpdaf').setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from astropy.table import join
from astroquery.sdss import SDSS

pf.housestyle_rcparams()

True

In [2]:
class QT_Candidates:
    def __init__(self, file_path: str = 'leadlines.csv'):
        self.file_path = file_path
        # self._data = {}
        self._keys = []
        self.analysed = False # ticker for coadd 
        self._initialise_file()
    
    def _initialise_file(self):
        self._leadlines = ascii.read('leadlines.csv')
        kyz = []
        for k in self._leadlines:
            kyz.append((k['dir'],k['key']))
        
        # Get unique (dir, key) tuples and sort them to ensure a consistent order.
        self._unique_kyz = sorted(list(set(kyz)))
        
        # Derive the simple keys from the sorted, unique list.
        self._keys = [k for (d,k) in self._unique_kyz]

        self._leadlines_OIII = self._leadlines[self._leadlines['Redshift']<0.8]

    def keys(self):
        return self._keys
    
    def get_candidate(self, key: str):
        global cand_leadlines
        cand_leadlines = self._leadlines_OIII[self._leadlines_OIII['key'] == key]

        key = cand_leadlines['key'][0]
        dir = cand_leadlines['dir'][0]
        loc = '/Volumes/Expansion/exp_thardy/'+dir+'/'+key+'_COMBINED_CUBE_MED_FINAL.fits'

        # with fits.open(loc) as hdul:
        #     hdr = hdul[0].header
        print(loc)
        hdr = Cube(loc).get_wcs_header()
        
        w = WCS(hdr)

        coords,wls = w.pixel_to_world(cand_leadlines['X_PEAK_SN'],
                                cand_leadlines['Y_PEAK_SN'],
                                cand_leadlines['Z_PEAK_SN'])
        
        cand_leadlines['ra'] = coords.ra.deg
        cand_leadlines['dec'] = coords.dec.deg
        cand_leadlines['OIII_est'] = (cand_leadlines['Redshift']+1)*5007

        crvals = hdr['CRVAL1'], hdr['CRVAL2']

        return cand_leadlines, cand_leadlines['zcluster'][0], crvals
        # redshift, table

    def analyse_all(self, raw=False, save_extras=False):
        self.analysed = True

        # Define output filenames
        output_filename = 'allsources.csv'
        raw_output_filename = 'allsources_uncorrected.csv'

        # Ensure a clean start by removing old files before the run begins.
        if os.path.exists(output_filename):
            os.remove(output_filename)
        if raw and os.path.exists(raw_output_filename):
            os.remove(raw_output_filename)

        # Store spectra and tables in memory to build final attributes
        spectra = {}
        all_tables = []
        all_raw_tables = []

        # Iterate over the unified list of unique (dir, key) tuples to avoid mismatches.
        for (d, k) in self._unique_kyz:
            name = k # This is the simple key name
            path = f'/Volumes/Expansion/exp_thardy/{d}/{k}_COMBINED_CUBE_MED_FINAL.fits'
            
            print(f'running {name}')
            try:
                tab, z, crvals = self.get_candidate(name)

                # Use the correct path that corresponds to the candidate's data (tab).
                indiv_cube = museCube(path, cluster_ra=crvals[0], cluster_dec=crvals[1])
                indiv_cube.process_multiple_candidates(tab, zcl=z)

                # --- Appending logic for the main table ---
                ex_table = indiv_cube.ex_table
                all_tables.append(ex_table)
                
                write_header = not os.path.exists(output_filename)
                s_buf = StringIO()
                ex_table.write(s_buf, format='csv')
                s_buf.seek(0)
                lines = s_buf.readlines()
                
                with open(output_filename, 'a') as f:
                    if write_header:
                        f.write(lines[0])
                    f.writelines(lines[1:])

                # --- Appending logic for the raw table (if requested) ---
                if raw:
                    raw_table = indiv_cube.raw_table
                    all_raw_tables.append(raw_table)
                    
                    write_header_raw = not os.path.exists(raw_output_filename)
                    s_buf_raw = StringIO()
                    raw_table.write(s_buf_raw, format='csv')
                    s_buf_raw.seek(0)
                    lines_raw = s_buf_raw.readlines()
                    
                    with open(raw_output_filename, 'a') as f_raw:
                        if write_header_raw:
                            f_raw.write(lines_raw[0])
                        f_raw.writelines(lines_raw[1:])
                
                if save_extras:
                    spectra[name] = indiv_cube.rest_spectra
            except:
                print('Failed, Skipping')

        # Create the combined table attributes from the in-memory lists
        if save_extras:
            self.combined_table = vstack(all_tables)
            if raw:
                self.combined_table_raw = vstack(all_raw_tables)

            self.spectra = spectra      

In [3]:
# class QT_Candidates:
#     def __init__(self, file_path: str = 'leadlines.csv'):
#         self.file_path = file_path
#         # self._data = {}
#         self._keys = []
#         self.analysed = False # ticker for coadd 
#         self._initialise_file()
    
#     def _initialise_file(self):
#         self._leadlines = ascii.read('leadlines.csv')
#         kyz = []
#         for k in self._leadlines:
#             kyz.append((k['dir'],k['key']))
        
#         # Get unique (dir, key) tuples and sort them to ensure a consistent order.
#         self._unique_kyz = sorted(list(set(kyz)))
        
#         # Derive the simple keys from the sorted, unique list.
#         self._keys = [k for (d,k) in self._unique_kyz]

#         self._leadlines_OIII = self._leadlines[self._leadlines['Redshift']<0.8]

#     def keys(self):
#         return self._keys
    
#     def get_candidate(self, key: str):
#         global cand_leadlines
#         cand_leadlines = self._leadlines_OIII[self._leadlines_OIII['key'] == key]

#         key = cand_leadlines['key'][0]
#         dir = cand_leadlines['dir'][0]
#         loc = '/Volumes/Expansion/exp_thardy/'+dir+'/'+key+'_COMBINED_CUBE_MED_FINAL.fits'

#         # with fits.open(loc) as hdul:
#         #     hdr = hdul[0].header
#         print(loc)
#         hdr = Cube(loc).get_wcs_header()
        
#         w = WCS(hdr)

#         coords,wls = w.pixel_to_world(cand_leadlines['X_PEAK_SN'],
#                                 cand_leadlines['Y_PEAK_SN'],
#                                 cand_leadlines['Z_PEAK_SN'])
        
#         cand_leadlines['ra'] = coords.ra.deg
#         cand_leadlines['dec'] = coords.dec.deg
#         cand_leadlines['OIII_est'] = (cand_leadlines['Redshift']+1)*5007

#         crvals = hdr['CRVAL1'], hdr['CRVAL2']

#         return cand_leadlines, cand_leadlines['zcluster'][0], crvals
#         # redshift, table

#     def analyse_all(self, raw=False):
#         self.analysed = True

#         tables = {}
#         raw_tables = {}
#         spectra = {}
#         # Iterate over the unified list of unique (dir, key) tuples to avoid mismatches.
#         for (d, k) in self._unique_kyz:
#             name = k # This is the simple key name
#             path = f'/Volumes/Expansion/exp_thardy/{d}/{k}_COMBINED_CUBE_MED_FINAL.fits'
            
#             print(f'running {name}')
#             global tab,z,crvals
#             tab, z, crvals = self.get_candidate(name)

#             # Use the correct path that corresponds to the candidate's data (tab).
#             indiv_cube = museCube(path, cluster_ra=crvals[0], cluster_dec=crvals[1])
#             indiv_cube.process_multiple_candidates(tab, zcl=z)

#             tables[name] = indiv_cube.ex_table
#             raw_tables[name] = indiv_cube.raw_table
#             spectra[name] = indiv_cube.rest_spectra

#         self.combined_table = vstack(list(tables.values()))
#         self.combined_table.write('allsources.csv', overwrite=True)

#         if raw:
#             self.combined_table_raw = vstack(list(raw_tables.values()))
#             self.combined_table_raw.write('allsources_uncorrected.csv', overwrite=True)

#         self.spectra = spectra

In [ ]:
cand = QT_Candidates()
cand.analyse_all()

running a141
/Volumes/Expansion/exp_thardy/cubes/a141_COMBINED_CUBE_MED_FINAL.fits
(np.float64(-24.644440638143095), np.float64(16.388710418951817))
(np.float64(-24.65210732218444), np.float64(16.386448867097652))
(np.float64(-24.645607181989106), np.float64(16.39292797199671))
Candidate 16.39292797199671,-24.645607181989106 generated an exception: 'float' object has no attribute 'copy'
Candidate 16.388710418951817,-24.644440638143095 generated an exception: 'float' object has no attribute 'copy'
Candidate 16.386448867097652,-24.65210732218444 generated an exception: 'float' object has no attribute 'copy'


(np.float64(-24.65005171519112), np.float64(16.39036091920871))
Candidate 16.39036091920871,-24.65005171519112 generated an exception: 'float' object has no attribute 'copy'
(np.float64(-24.640107221096084), np.float64(16.380886857140663))
(np.float64(-24.64338485946044), np.float64(16.39494490055822))


In [ ]:
cand_leadlines